## Baseline YOLO11 Training

This notebook trains the first YOLO11 baseline model for parasite object detection.  
The goal is to establish a simple reference result before running improvement experiments.

### Install training dependencies

This cell installs the Ultralytics package used to train and evaluate YOLO11 models.  
The project uses Ultralytics because it provides a practical training pipeline for YOLO object detection.

In [ ]:
!pip install ultralytics

### Verify training environment

This cell imports the training library and checks the available runtime environment.  
This confirms that YOLO11 can be loaded before starting baseline training.

In [ ]:
from ultralytics import YOLO
import torch
from pathlib import Path

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("GPU not available. Training may be slow on CPU.")

### Set training dataset configuration

This cell defines the path to the YOLO dataset configuration file used for training.  
The baseline model will use this file to locate the parasite dataset splits and class names.

In [ ]:
DATA_DIR = Path("../data")
DATA_YAML = DATA_DIR / "data.yaml"

print("data.yaml exists:", DATA_YAML.exists())
print("data.yaml path:", DATA_YAML.resolve())

### Load baseline YOLO11 model

This cell loads the smallest YOLO11 detection model as the baseline.  
A lightweight baseline is useful for verifying the training workflow before testing larger or more expensive models.

In [ ]:
model = YOLO("yolo11n.pt")

print("Baseline model loaded successfully.")

### Train baseline model

This cell trains the YOLO11n baseline model using simple training settings.  
The baseline creates a reference result that later experiments can be compared against.

In [ ]:
baseline_results = model.train(
    data=str(DATA_YAML),
    epochs=50,
    imgsz=640,
    batch=16,
    device=0,
    project="runs",
    name="yolo11n_baseline",
    seed=42
)

### Validate baseline model

This cell evaluates the trained YOLO11n baseline model on the validation split.  
Validation metrics provide the first reference point for later improvement experiments.

In [ ]:
baseline_metrics = model.val(
    data=str(DATA_YAML),
    imgsz=640,
    batch=16,
    device=0,
    project="runs",
    name="yolo11n_baseline_val"
)

### Summarize baseline validation metrics

This cell extracts the main validation metrics from the baseline YOLO11n run.  
These metrics will be used as the reference point for later improvement experiments.

In [ ]:
baseline_summary = {
    "model": "YOLO11n",
    "imgsz": 640,
    "precision": float(baseline_metrics.box.mp),
    "recall": float(baseline_metrics.box.mr),
    "mAP50": float(baseline_metrics.box.map50),
    "mAP50-95": float(baseline_metrics.box.map)
}

baseline_summary_df = pd.DataFrame([baseline_summary])
baseline_summary_df

### Locate baseline validation outputs

This cell identifies where YOLO saved the baseline validation artifacts.  
These files are used to inspect training quality beyond the summary metrics.

In [ ]:
from pathlib import Path

val_dir = Path("runs/detect/yolo11n_baseline_val")

print("Validation output folder exists:", val_dir.exists())

for file_path in val_dir.iterdir():
    print(file_path.name)

### Display baseline validation plots

This cell displays the main validation artifacts generated by YOLO.  
These plots help inspect class confusion, precision-recall behavior, and prediction quality beyond the summary metrics.

In [ ]:
from IPython.display import Image, display

plot_files = [
    "confusion_matrix.png",
    "confusion_matrix_normalized.png",
    "BoxPR_curve.png",
    "BoxF1_curve.png",
    "BoxP_curve.png",
    "BoxR_curve.png",
]

for file_name in plot_files:
    file_path = val_dir / file_name

    if file_path.exists():
        print(file_name)
        display(Image(filename=str(file_path)))
    else:
        print(f"Missing: {file_name}")

### Save baseline summary

This cell saves the baseline validation metrics to a CSV file.  
The saved table makes it easier to compare this baseline against later experiments.

In [ ]:
baseline_summary_df.to_csv("/kaggle/working/runs/detect/runs/yolo11n_baseline_summary.csv", index=False)

print("Saved baseline summary to yolo11n_baseline_summary.csv")